# Policy System ADLS Volume Setup

One-time environment setup. **Not part of the scheduled job.** Safe to re-run.

**Prerequisites (created once in the UI / Catalog Explorer):**
1. Azure Databricks Access Connector with *Storage Blob Data Contributor* on `insurancepolicy1adls`
2. Storage credential pointing to that Access Connector
3. External locations for the three containers, for example:

```sql
-- CREATE EXTERNAL LOCATION IF NOT EXISTS bronze_loc URL 'abfss://bronze@insurancepolicy1adls.dfs.core.windows.net/' WITH (STORAGE CREDENTIAL <your_credential>);
-- CREATE EXTERNAL LOCATION IF NOT EXISTS silver_loc URL 'abfss://silver@insurancepolicy1adls.dfs.core.windows.net/' WITH (STORAGE CREDENTIAL <your_credential>);
-- CREATE EXTERNAL LOCATION IF NOT EXISTS gold_loc   URL 'abfss://gold@insurancepolicy1adls.dfs.core.windows.net/'   WITH (STORAGE CREDENTIAL <your_credential>);
```

**Layout rule:** a volume and an external table may not overlap the same path, so every volume lives in a
`files/` subfolder and external Delta tables live next to it (for example `silver/customer`).

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS policysystemdatabricks.bronze;
CREATE SCHEMA IF NOT EXISTS policysystemdatabricks.silver;
CREATE SCHEMA IF NOT EXISTS policysystemdatabricks.gold;

In [0]:
# Widget: moving the bronze volume changes where raw files must land (bronze/files/...),
# so it only happens when you explicitly set this to YES after reading the diagnostics below.
dbutils.widgets.dropdown("move_bronze_volume", "NO", ["NO", "YES"], "Move bronze volume to files/")

CATALOG = "policysystemdatabricks"
ACCOUNT = "insurancepolicy1adls"
VOLUMES = {"bronze": "bronzelayer", "silver": "silverlayer", "gold": "goldlayer"}


def target_location(layer: str) -> str:
    return f"abfss://{layer}@{ACCOUNT}.dfs.core.windows.net/files/"


def current_volume_location(layer: str, volume: str):
    """Returns the volume's storage location, or None if the volume does not exist."""
    if spark.sql(f"SHOW VOLUMES IN {CATALOG}.{layer} LIKE '{volume}'").count() == 0:
        return None
    row = spark.sql(f"DESCRIBE VOLUME {CATALOG}.{layer}.{volume}").first().asDict()
    return next(v for k, v in row.items() if "location" in k.lower())

In [0]:
# Any external table whose location sits under a volume's location is an overlap -> writes to it fail.
for layer, volume in VOLUMES.items():
    print(f"{layer:7s} volume {volume:12s} -> {current_volume_location(layer, volume)}")

print()
for layer in VOLUMES:
    for t in spark.sql(f"SHOW TABLES IN {CATALOG}.{layer}").collect():
        if t["isTemporary"]:
            continue
        detail = spark.sql(f"DESCRIBE DETAIL {CATALOG}.{layer}.{t['tableName']}").first()
        print(f"{layer:7s} table  {t['tableName']:20s} -> {detail['location']}")

bronze  volume bronzelayer  -> abfss://bronze@insurancepolicy1adls.dfs.core.windows.net/files
silver  volume silverlayer  -> abfss://silver@insurancepolicy1adls.dfs.core.windows.net/files
gold    volume goldlayer    -> abfss://gold@insurancepolicy1adls.dfs.core.windows.net/files

bronze  table  agent                -> 
bronze  table  branch               -> 
bronze  table  claim                -> 
bronze  table  customer             -> 
bronze  table  policydata           -> 
silver  table  agent                -> abfss://silver@insurancepolicy1adls.dfs.core.windows.net/agent
silver  table  branch               -> abfss://silver@insurancepolicy1adls.dfs.core.windows.net/branch
silver  table  claim                -> abfss://silver@insurancepolicy1adls.dfs.core.windows.net/claim
silver  table  customer             -> abfss://silver@insurancepolicy1adls.dfs.core.windows.net/customer
silver  table  policydata           -> abfss://silver@insurancepolicy1adls.dfs.core.windows.net/policydata


In [0]:
# Dropping an EXTERNAL volume only removes the Unity Catalog object; the files in ADLS are not deleted.
move_bronze = dbutils.widgets.get("move_bronze_volume") == "YES"

for layer, volume in VOLUMES.items():
    fq_name = f"{CATALOG}.{layer}.{volume}"
    target = target_location(layer)
    current = current_volume_location(layer, volume)

    if current is not None and current.rstrip("/") == target.rstrip("/"):
        print(f"{fq_name}: already at {target}")
        continue
    if layer == "bronze" and current is not None and not move_bronze:
        print(f"{fq_name}: left at {current} (set move_bronze_volume = YES to move it)")
        continue

    if current is not None:
        spark.sql(f"DROP VOLUME IF EXISTS {fq_name}")
    spark.sql(f"CREATE EXTERNAL VOLUME {fq_name} LOCATION '{target}'")
    print(f"{fq_name}: created at {target}")

policysystemdatabricks.bronze.bronzelayer: already at abfss://bronze@insurancepolicy1adls.dfs.core.windows.net/files/
policysystemdatabricks.silver.silverlayer: already at abfss://silver@insurancepolicy1adls.dfs.core.windows.net/files/
policysystemdatabricks.gold.goldlayer: already at abfss://gold@insurancepolicy1adls.dfs.core.windows.net/files/


In [0]:
for layer, volume in VOLUMES.items():
    path = f"/Volumes/{CATALOG}/{layer}/{volume}/"
    print(layer, [f.name for f in dbutils.fs.ls(path)])

bronze []
silver []
gold []
